## Interacting with files

In [ ]:
from cirro import DataPortal

portal = DataPortal(base_url="")

Find the file you are looking for by defining the project and dataset, then using `read_file` or `read_files` to read file contents directly into Python objects.

The file format is inferred automatically from the extension (`.csv`, `.tsv`, `.json`, `.parquet`, `.feather`, `.pkl`, `.xlsx`, `.h5ad`), or can be specified with the `format` parameter.

### Inspecting files

In [2]:
# Get the project which contains the dataset
project = portal.get_project_by_name("Pipeline Development")

# Get the dataset of interest based on its name
dataset = project.get_dataset("Genomic variant calling - parameter validation")

print(f"Dataset: {dataset.name}")
print(f"Files: {len(dataset.list_files()):,}")
for file in dataset.list_files():
    if file.name.endswith('.vcf.gz'):
        print(file.name)

Dataset: Genomic variant calling - parameter validation
Files: 235
data/variant_calling/haplotypecaller/ERR031935/ERR031935.haplotypecaller.filtered.vcf.gz
data/variant_calling/haplotypecaller/ERR031935/ERR031935.haplotypecaller.vcf.gz
data/variant_calling/strelka/ERR031935/ERR031935.strelka.genome.vcf.gz
data/variant_calling/strelka/ERR031935/ERR031935.strelka.variants.vcf.gz


### Reading a file

Read a single file into a DataFrame using `read_file`. The tab-separated format is specified explicitly with `sep='\t'`.

In [3]:
# Read a single file matched by a glob pattern
df = dataset.read_file(glob="*.variants.vcf.gz", filetype="csv", sep="\t", comment="#", header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9
0,chr20,60826,.,T,A,1,LowDepth;LowGQX;NoPassedVariantGTs,MQ=17;SNVHPOL=2,GT:GQ:GQX:DP:DPF:AD:ADF:ADR:SB:FT:PL,"0/1:3:0:1:0:0,1:0,1:0,0:0.0:LowGQX;LowDepth:29..."
1,chr20,60850,.,A,T,1,LowDepth;LowGQX;NoPassedVariantGTs,MQ=24;SNVHPOL=4,GT:GQ:GQX:DP:DPF:AD:ADF:ADR:SB:FT:PL,"0/1:3:1:1:0:0,1:0,1:0,0:0.0:LowGQX;LowDepth:30..."
2,chr20,62437,.,C,T,3,LowDepth;LowGQX;NoPassedVariantGTs,MQ=22;SNVHPOL=2,GT:GQ:GQX:DP:DPF:AD:ADF:ADR:SB:FT:PL,"0/1:3:0:1:1:0,1:0,0:0,1:0.0:LowGQX;LowDepth:35..."
3,chr20,62467,.,C,A,4,LowDepth;LowGQX;NoPassedVariantGTs,MQ=24;SNVHPOL=2,GT:GQ:GQX:DP:DPF:AD:ADF:ADR:SB:FT:PL,"0/1:3:0:1:0:0,1:0,0:0,1:0.0:LowGQX;LowDepth:36..."
4,chr20,62469,.,G,A,3,LowDepth;LowGQX;NoPassedVariantGTs,MQ=24;SNVHPOL=3,GT:GQ:GQX:DP:DPF:AD:ADF:ADR:SB:FT:PL,"0/1:3:0:1:0:0,1:0,0:0,1:0.0:LowGQX;LowDepth:34..."


### Reading multiple files

Use `read_files` to iterate over multiple matching files. With `{name}` capture placeholders in the `pattern`, extracted values are returned alongside each file's content.

In [4]:
# Extract folder names from the path automatically using {name} placeholders
for df, meta in dataset.read_files(
    pattern="*/strelka/{sample}/*.strelka.{type}.vcf.gz",
    filetype="csv",
    sep="\t",
    comment="#",
    header=None
):
    print(meta, df.shape)

{'sample': 'ERR031935', 'type': 'genome'} (790381, 10)
{'sample': 'ERR031935', 'type': 'variants'} (36318, 10)


You can also view any artifacts produced by running the analysis, such as the workflow report, graph, or logs.

### Getting metadata

In [6]:
# Reading nextflow trace file
trace = dataset.get_trace()
trace.head()

,task_id,hash,native_id,name,status,exit,submit,duration,realtime,%cpu,peak_rss,peak_vmem,rchar,wchar
0,7,fb/18dde6,4a268ebd-7d6d-42e7-8753-a9ee3f0b1aca,NFCORE_SAREK:SAREK:PREPARE_INTERVALS:TABIX_BGZ...,COMPLETED,0,2023-08-29 18:55:47.794,2m 52s,0ms,79.1%,3 MB,5.4 MB,79.8 KB,3.6 KB
1,6,e0/e394ac,4195506d-60cd-4771-ac03-e801adeb7794,NFCORE_SAREK:SAREK:PREPARE_INTERVALS:CREATE_IN...,COMPLETED,0,2023-08-29 18:55:47.807,2m 53s,0ms,171.4%,2.9 MB,11 MB,43 KB,1.9 KB
2,21,57/3dfeca,563917fe-c79c-419c-a4c8-081457e9241a,NFCORE_SAREK:SAREK:PREPARE_INTERVALS:TABIX_BGZ...,COMPLETED,0,2023-08-29 18:58:41.177,38.5s,0ms,82.8%,3.1 MB,5.4 MB,79 KB,3.2 KB
3,23,37/ebcc21,27342717-5e4c-4ce3-81bd-5dd04192c7a7,NFCORE_SAREK:SAREK:PREPARE_INTERVALS:TABIX_BGZ...,COMPLETED,0,2023-08-29 18:58:41.319,38.7s,0ms,86.7%,3.1 MB,5.4 MB,79 KB,3.2 KB
4,20,09/9937ff,4324507a-80a9-4957-9b99-6de4a949fd62,NFCORE_SAREK:SAREK:PREPARE_INTERVALS:TABIX_BGZ...,COMPLETED,0,2023-08-29 18:58:41.352,39.1s,0ms,84.7%,3.1 MB,5.4 MB,79 KB,3.2 KB


In [7]:
# Get the logs
logs = dataset.get_logs()
logs.split("\n")[:10]

['PW_WORKFLOW_SCRIPT=main.nf',
 'PW_AWS_REGION=us-west-2',
 'PW_BATCH_JOB_ROLE=arn:aws:iam::523221283927:role/Cirro-BatchJobRole-9a31492a',
 'PW_S3_TRANSFORM_WORKFLOW=s3://pubweb-resources-develop/process/hutch/data-transforms/workflow/',
 'PW_ONDEMAND_JOB_QUEUE=arn:aws:batch:us-west-2:523221283927:job-queue/Cirro-OnDemand-9a31492a',
 'PW_DATASET=3fb7e8f8-b62d-43a6-ad08-eb28f59bd141',
 'PW_WORKFLOW_VERSION=3.2.3',
 'PW_S3_RESTORE_SESSION_DIR=s3://project-9a31492a-e679-43ce-9f06-d84213c8f7f7-scratch/workdir/session/acc47b45-df28-4120-ab0d-7106ba7c5fc4',
 'PW_SPOT_JOB_QUEUE=arn:aws:batch:us-west-2:523221283927:job-queue/Cirro-Spot-9a31492a',
 'PW_S3_DATASET=s3://project-9a31492a-e679-43ce-9f06-d84213c8f7f7/datasets/3fb7e8f8-b62d-43a6-ad08-eb28f59bd141']